# Credit Card / Online Payments Fraud Detection

End-to-end exploration and modeling notebook:
1. EDA & class-imbalance visualization
2. Feature engineering
3. Sampling strategies (undersample / oversample / SMOTE)
4. Random Forest & XGBoost models
5. Evaluation — precision, recall, ROC-AUC, PR-AUC
6. Precision/recall tradeoff & business decision thresholds

> This notebook mirrors `src/fraud_detection.py`. For production use or re-running headlessly, prefer the script — it's the same logic, parameterized via CLI flags.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 1. Load the data

In [ ]:
DATA_PATH = "../data/AIML_Dataset.csv"  # update if your file has a different name
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

In [ ]:
df.columns

## 2. Exploratory Data Analysis (EDA)

In [ ]:
df["isFraud"].value_counts()

In [ ]:
df["isFlaggedFraud"].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
fraud_pct = round((df["isFraud"].value_counts()[1] / df.shape[0]) * 100, 4)
print(f"Fraud rate: {fraud_pct}% of all transactions")

### Class imbalance

In [ ]:
fraud_counts = df["isFraud"].value_counts()
plt.figure(figsize=(5,4))
ax = sns.barplot(x=fraud_counts.index.astype(str), y=fraud_counts.values,
                  palette=["#2c7fb8", "#d7191c"])
ax.set_xticklabels(["Legit (0)", "Fraud (1)"])
for i, v in enumerate(fraud_counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.title(f"Class Imbalance — Fraud is only {fraud_pct}% of transactions")
plt.ylabel("Count")
plt.show()

### Transaction types

In [ ]:
df["type"].value_counts().plot(kind="bar", title="Transaction Types", color="darkgreen")
plt.xlabel("Transaction Type")
plt.ylabel("Count")
plt.show()

In [ ]:
fraud_by_type = df.groupby("type")["isFraud"].mean().sort_values(ascending=False)
fraud_by_type.plot(kind="bar", title="Fraud Rate by Type", color="salmon")
plt.ylabel("Fraud Rate")
plt.show()
fraud_by_type

### Transaction amount

In [ ]:
df["amount"].describe().astype(int)

In [ ]:
sns.histplot(np.log1p(df["amount"]), bins=100, kde=True, color="green")
plt.title("Transaction Amount Distribution (log scale)")
plt.xlabel("Log(Amount + 1)")
plt.show()

In [ ]:
sns.boxplot(data=df[df["amount"] < 50000], x="isFraud", y="amount")
plt.title("Amount vs isFraud (Filtered under 50k)")
plt.show()

### Balance-based features

In [ ]:
df["balanceDiffOrig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["balanceDiffDest"] = df["newbalanceDest"] - df["oldbalanceDest"]

In [ ]:
print((df["balanceDiffOrig"] < 0).sum(), "rows with a negative origin balance diff")
print((df["balanceDiffDest"] < 0).sum(), "rows with a negative destination balance diff")

In [ ]:
df.head(2)

### Frauds over time

In [ ]:
frauds_per_step = df[df["isFraud"] == 1]["step"].value_counts().sort_index()
plt.plot(frauds_per_step.index, frauds_per_step.values, color="crimson")
plt.xlabel("Step (Time)")
plt.ylabel("Number of Frauds")
plt.title("Frauds Over Time")
plt.grid(True)
plt.show()

### Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(8,6))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (numeric features)")
plt.show()

## 3. Feature engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()
df_model["origBalanceMismatch"] = (df_model["balanceDiffOrig"] < 0).astype(int)
df_model["destBalanceMismatch"] = (df_model["balanceDiffDest"] < 0).astype(int)

le = LabelEncoder()
df_model["type_encoded"] = le.fit_transform(df_model["type"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

df_model = df_model.drop(columns=["nameOrig", "nameDest", "type"])
df_model.head()

## 4. Train / test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=["isFraud"])
y = df_model["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts())

## 5. Handling class imbalance — sampling strategies

We try three approaches on the **training set only** (never resample the test set — that would leak information and give an unrealistically optimistic evaluation):

- **Random undersampling** of the majority (legit) class
- **Random oversampling** of the minority (fraud) class
- **SMOTE** (Synthetic Minority Oversampling Technique)

In [ ]:
from sklearn.utils import resample

def undersample(X_train, y_train, random_state=42):
    df_train = pd.concat([X_train, y_train], axis=1)
    fraud = df_train[df_train["isFraud"] == 1]
    legit = df_train[df_train["isFraud"] == 0]
    legit_down = resample(legit, replace=False, n_samples=len(fraud), random_state=random_state)
    df_bal = pd.concat([fraud, legit_down])
    return df_bal.drop(columns=["isFraud"]), df_bal["isFraud"]

def oversample(X_train, y_train, random_state=42):
    df_train = pd.concat([X_train, y_train], axis=1)
    fraud = df_train[df_train["isFraud"] == 1]
    legit = df_train[df_train["isFraud"] == 0]
    fraud_up = resample(fraud, replace=True, n_samples=len(legit), random_state=random_state)
    df_bal = pd.concat([legit, fraud_up])
    return df_bal.drop(columns=["isFraud"]), df_bal["isFraud"]

In [ ]:
# SMOTE (requires `pip install imbalanced-learn`)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_train_smote.value_counts().to_dict())

We'll move forward with **SMOTE** for the main models below, since it tends to generalize better than naive oversampling and keeps more information than undersampling. Feel free to swap in `undersample(...)` or `oversample(...)` to compare.

## 6. Model training — Random Forest & XGBoost

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=2, n_jobs=-1,
    class_weight="balanced_subsample", random_state=42
)
rf.fit(X_train_smote, y_train_smote)

In [ ]:
from xgboost import XGBClassifier

neg, pos = np.bincount(y_train_smote)
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9, eval_metric="auc",
    scale_pos_weight=neg/max(pos,1), random_state=42, n_jobs=-1
)
xgb.fit(X_train_smote, y_train_smote)

## 7. Evaluation — precision, recall, ROC-AUC

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              average_precision_score, precision_recall_curve, roc_curve)

def evaluate(name, model, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, digits=4))
    print("ROC-AUC:", roc_auc_score(y_test, proba))
    print("PR-AUC :", average_precision_score(y_test, proba))

    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Legit","Fraud"], yticklabels=["Legit","Fraud"])
    plt.title(f"Confusion Matrix — {name}")
    plt.ylabel("Actual"); plt.xlabel("Predicted")
    plt.show()
    return proba

rf_proba = evaluate("Random Forest", rf, X_test, y_test)
xgb_proba = evaluate("XGBoost", xgb, X_test, y_test)

In [ ]:
plt.figure(figsize=(6,5))
for name, proba in [("Random Forest", rf_proba), ("XGBoost", xgb_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
plt.plot([0,1],[0,1],"k--", alpha=0.4)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.show()

In [ ]:
plt.figure(figsize=(6,5))
for name, proba in [("Random Forest", rf_proba), ("XGBoost", xgb_proba)]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    plt.plot(rec, prec, label=f"{name} (PR-AUC={ap:.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall Curve"); plt.legend(); plt.show()

## 8. Precision vs. Recall tradeoff & business decision thresholds

In fraud detection:
- **High recall, low precision** → catch almost every fraud, but flood analysts with false alarms (expensive in review time, annoying for legit customers who get blocked).
- **High precision, low recall** → only flag transactions you're very confident about, but you let more real fraud slip through (expensive in direct losses + chargebacks).

The right threshold is a **business decision**, not a purely technical one. Below we sweep the threshold and translate it into a rough cost model:

- `fraud_cost`: estimated $ lost when a fraud is missed (false negative)
- `review_cost`: estimated $ cost of manually reviewing one flagged transaction (true positive or false positive)

Adjust these two numbers to match your real business economics.

In [ ]:
def business_threshold_report(y_test, proba, fraud_cost=500, review_cost=5):
    rows = []
    for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0,1]).ravel()
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        flagged = tp + fp
        cost = fn * fraud_cost + flagged * review_cost
        rows.append({"threshold": t, "precision": round(precision,3), "recall": round(recall,3),
                     "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                     "flagged_for_review": flagged, "estimated_cost": round(cost,2)})
    report = pd.DataFrame(rows)
    best = report.loc[report["estimated_cost"].idxmin()]
    print(f"Lowest-cost threshold: {best['threshold']} "
          f"(precision={best['precision']}, recall={best['recall']}, cost={best['estimated_cost']})")
    return report

rf_report = business_threshold_report(y_test, rf_proba)
rf_report

In [ ]:
plt.plot(rf_report["threshold"], rf_report["estimated_cost"], marker="o", color="purple")
best_t = rf_report.loc[rf_report["estimated_cost"].idxmin(), "threshold"]
plt.axvline(best_t, color="red", linestyle="--", label=f"min-cost threshold = {best_t}")
plt.xlabel("Decision Threshold"); plt.ylabel("Estimated Business Cost")
plt.title("Business Cost vs Threshold — Random Forest")
plt.legend(); plt.show()

## 9. Save the model

In [ ]:
import joblib
import os
os.makedirs("../outputs/models", exist_ok=True)
joblib.dump(rf, "../outputs/models/random_forest.joblib")
joblib.dump(xgb, "../outputs/models/xgboost.joblib")
print("Models saved.")

## 10. Conclusions

- The dataset is **heavily imbalanced** — fraud is a tiny fraction of all transactions, so accuracy alone is a misleading metric.
- **SMOTE** (or oversampling) on the training data substantially improves the model's ability to learn fraud patterns versus training on the raw imbalanced data.
- **Random Forest** and **XGBoost** both perform well; compare their ROC-AUC / PR-AUC above and pick whichever generalizes better on your holdout set.
- There's no single 'best' classification threshold — it depends on the relative cost of **missing a fraud** vs. the cost of **manually reviewing a flagged transaction**. Use the business-cost sweep above to pick an operating point that matches your organization's actual cost structure, and revisit it periodically as those costs change.

For a reusable, scriptable version of this whole pipeline (with CLI flags for sampling strategy, model choice, and cost assumptions), see `src/fraud_detection.py`.